# Week 3 - QLoRA fine-tune of SmolLM3-3B for log triage

**Before running** (right-hand panel, *Session options*):
1. **Accelerator: GPU T4 x2** (not P100). The notebook uses one of the two T4s.
2. **Internet: On** (needs a phone-verified Kaggle account).
3. **Add Input** -> your private dataset that contains `train_sft.jsonl` and `dev_sft.jsonl`.

**How to run**
1. First time: leave `SMOKE_TEST = True` and click **Run All**. About 15-25 minutes. It trains
   10 steps, then merges and converts to GGUF - checking the whole pipeline end to end.
2. If every cell finishes without a red error: set `SMOKE_TEST = False`, then
   **Save Version -> Save & Run All (Commit)**. It runs in the background; you can close the tab.
3. When the version finishes: open it -> **Output** -> download
   `smollm3-logtriage-q8_0.gguf`, `run_info.json`, `train_log.json`.

The prompt text comes from your frozen `config.build_prompt()` (already inside the jsonl files).
Nothing in this notebook touches the eval set. The checkpoint is chosen by **dev loss only**.

In [ ]:
# 1) Install the exact library versions this notebook was tested with.
#    Red "ERROR: pip's dependency resolver ..." lines about OTHER packages are normal on Kaggle - ignore them.
%pip install -q "transformers==5.17.0" "peft==0.21.0" "accelerate==1.15.0" "bitsandbytes==0.50.2" "sentencepiece==0.2.2"

In [ ]:
# 2) Settings. The only line you normally change is SMOKE_TEST.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # use ONE T4 (two T4s adds overhead, not speed, at this size)

SMOKE_TEST = True          # True = quick 10-step check of the whole pipeline. False = the real run.

MODEL_ID  = "HuggingFaceTB/SmolLM3-3B"
DATA_DIR  = "/kaggle/input"
OUT       = "/kaggle/working"
LLAMA_CPP_COMMIT = "b23efaa2ef147f547ee75cbf0c621d61904de80e"   # pinned GGUF converter version
# The base model's small non-weight files. They are copied unchanged into the merged model,
# so the fine-tuned GGUF differs from the base model ONLY in its weights.
META_FILES = ["config.json", "generation_config.json", "tokenizer.json", "tokenizer_config.json",
              "special_tokens_map.json", "chat_template.jinja"]

# Training settings - fixed BEFORE training. Picked from common QLoRA defaults, not tuned on eval.
EPOCHS        = 3
LEARNING_RATE = 2e-4
BATCH         = 4          # examples per step on the GPU
GRAD_ACCUM    = 4          # steps added together -> effective batch 16
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
MAX_LEN       = 1024       # longer examples are dropped (and counted)
SEED          = 42

In [ ]:
# 3) Check the machine and find the data.
import gc, glob, json, math, random, subprocess, sys, time
import torch, transformers, peft, accelerate

print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| accelerate", accelerate.__version__)
assert torch.cuda.is_available(), "No GPU. Session options -> Accelerator -> GPU T4 x2"
GPU = torch.cuda.get_device_name(0)
print("GPU:", GPU)
if torch.cuda.get_device_capability(0) < (7, 5):
    raise SystemExit("This notebook is set up for a T4. Session options -> Accelerator -> GPU T4 x2")

def find(name):
    hits = glob.glob(f"{DATA_DIR}/**/{name}", recursive=True)
    if len(hits) != 1:
        raise SystemExit(f"Expected exactly one {name} under {DATA_DIR}, found {hits}. Add your dataset as Input.")
    return hits[0]

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

import hashlib
def sha256(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()

TRAIN_FILE, DEV_FILE = find("train_sft.jsonl"), find("dev_sft.jsonl")
train_raw = read_jsonl(TRAIN_FILE)
dev_raw   = read_jsonl(DEV_FILE)
DATA_SHA256 = {"train_sft.jsonl": sha256(TRAIN_FILE), "dev_sft.jsonl": sha256(DEV_FILE)}
print(f"train {len(train_raw)} | dev {len(dev_raw)}")
if SMOKE_TEST:
    train_raw, dev_raw = train_raw[:64], dev_raw[:16]
    print("SMOKE TEST: using", len(train_raw), "train and", len(dev_raw), "dev examples")
random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# 4) Will Ollama (llama.cpp) put a start-of-text (BOS) token in front of every prompt?
#    Training must feed the model exactly what Ollama will feed it at scoring time.
#    We ask the same pinned converter that will make our GGUF, using only the base model's small files.
from huggingface_hub import snapshot_download

def run(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    if r.returncode != 0:
        raise SystemExit("Command failed - paste the output above to Claude.")

LLAMA = "/tmp/llama.cpp"
if not os.path.exists(f"{LLAMA}/convert_hf_to_gguf.py"):
    run(f"git init -q {LLAMA} && cd {LLAMA} && git remote add origin https://github.com/ggml-org/llama.cpp "
        f"&& git fetch -q --depth 1 origin {LLAMA_CPP_COMMIT} && git checkout -q FETCH_HEAD")
sys.path.insert(0, f"{LLAMA}/gguf-py")
import gguf

BASE_FILES = snapshot_download(MODEL_ID, allow_patterns=META_FILES)
run(f"{sys.executable} {LLAMA}/convert_hf_to_gguf.py {BASE_FILES} --vocab-only --outfile /tmp/base-vocab.gguf")

def gguf_meta(path):
    return {k: f.contents() for k, f in gguf.GGUFReader(path).fields.items() if not k.startswith("GGUF.")}

BASE_META = gguf_meta("/tmp/base-vocab.gguf")
vocab_type, pre = BASE_META.get("tokenizer.ggml.model"), BASE_META.get("tokenizer.ggml.pre")
# llama.cpp's rules (src/llama-vocab.cpp at the pinned commit): a flag in the file wins; otherwise a default.
LLAMA3_STYLE = {"llama3", "llama-v3", "llama-bpe", "falcon3", "falcon-h1", "pixtral", "midm-2.0",
                "lfm2", "jina-v5-nano", "tekken", "chameleon"}
if "tokenizer.ggml.add_bos_token" in BASE_META:
    ADD_BOS, BOS_WHY = bool(BASE_META["tokenizer.ggml.add_bos_token"]), "the GGUF file says so"
elif vocab_type == "gpt2":
    ADD_BOS, BOS_WHY = pre in LLAMA3_STYLE, f"llama.cpp default for pre-tokenizer '{pre}'"
elif vocab_type == "llama":
    ADD_BOS, BOS_WHY = True, "llama.cpp default for SentencePiece vocabularies"
else:
    raise SystemExit(f"Unexpected vocabulary type {vocab_type!r} - paste this to Claude.")
BOS_ID = BASE_META.get("tokenizer.ggml.bos_token_id")
if ADD_BOS and BOS_ID is None:
    raise SystemExit("llama.cpp wants a BOS token but the file names none - paste this to Claude.")
print(f"vocab type {vocab_type}, pre-tokenizer {pre}")
print(f"==> Ollama will {'ADD' if ADD_BOS else 'NOT add'} a BOS token ({BOS_WHY}). Training does the same.")

In [ ]:
# 5) Tokenise. The model sees the prompt but is only graded (loss) on the answer + <|im_end|>.
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
IM_END = tok.convert_tokens_to_ids("<|im_end|>")
assert isinstance(IM_END, int) and IM_END != tok.unk_token_id, "<|im_end|> is not a single token"
assert IM_END in tok("<|im_start|>user\nx<|im_end|>", add_special_tokens=False)["input_ids"], \
    "tokenizer does not recognise <|im_end|> inside text"
PAD = tok.pad_token_id if tok.pad_token_id is not None else IM_END
print("<|im_end|> id:", IM_END, "| pad id:", PAD)
# The GGUF vocabulary must agree with the tokenizer used for training.
assert BASE_META["tokenizer.ggml.tokens"][IM_END] == "<|im_end|>", "GGUF and tokenizer disagree on <|im_end|>"

def prompt_ids(text):
    return ([BOS_ID] if ADD_BOS else []) + tok(text, add_special_tokens=False)["input_ids"]

def encode(ex):
    p = prompt_ids(ex["prompt"])
    a = tok(ex["answer"], add_special_tokens=False)["input_ids"] + [IM_END]
    return {"input_ids": p + a, "labels": [-100] * len(p) + a}

def encode_all(rows, name):
    out, dropped = [], 0
    for ex in rows:
        e = encode(ex)
        if len(e["input_ids"]) > MAX_LEN:
            dropped += 1
            continue
        out.append(e)
    lens = sorted(len(e["input_ids"]) for e in out)
    print(f"{name}: {len(out)} examples, dropped {dropped} longer than {MAX_LEN} tokens | "
          f"tokens median {lens[len(lens) // 2]}, max {lens[-1]}")
    return out, dropped

train_enc, train_dropped = encode_all(train_raw, "train")
dev_enc, dev_dropped = encode_all(dev_raw, "dev")

# Show what the model is actually graded on, for one example.
ex = train_enc[0]
graded = [t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100]
print("\nGraded part of example 0 (everything before it is prompt, not graded):")
print(repr(tok.decode(graded)))
assert tok.decode(graded) == train_raw[0]["answer"] + "<|im_end|>"

# Cross-check for later: Ollama reports how many prompt tokens it used. For this dev example
# it should report exactly this number (see the instructions Claude gave you).
CHECK = {"dev_id": dev_raw[0]["id"], "prompt_tokens": len(prompt_ids(dev_raw[0]["prompt"]))}
print(f"\nCROSS-CHECK: dev example {CHECK['dev_id']} -> {CHECK['prompt_tokens']} prompt tokens as trained")

In [ ]:
# 6) Load SmolLM3 in 4-bit and attach the LoRA adapter (this is the "Q" in QLoRA).
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                             device_map={"": 0}, dtype=torch.float16)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                        gradient_checkpointing_kwargs={"use_reentrant": False})

In [ ]:
# 7) LoRA settings.
lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
                  target_modules="all-linear", task_type="CAUSAL_LM")
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# 8) Train. Dev loss is measured after every epoch; the epoch with the LOWEST dev loss is kept.
from transformers import Trainer, TrainingArguments, TrainerCallback

class ListDataset(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collate(batch):
    n = max(len(x["input_ids"]) for x in batch)
    return {
        "input_ids": torch.tensor([x["input_ids"] + [PAD] * (n - len(x["input_ids"])) for x in batch]),
        "labels": torch.tensor([x["labels"] + [-100] * (n - len(x["labels"])) for x in batch]),
        "attention_mask": torch.tensor([[1] * len(x["input_ids"]) + [0] * (n - len(x["input_ids"]))
                                        for x in batch]),
    }

class StopIfNaN(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs and not math.isfinite(logs["loss"]):
            print("\n!!! Loss became NaN/inf - stopping. Paste this output to Claude.")
            control.should_training_stop = True

every = "steps" if SMOKE_TEST else "epoch"
args = TrainingArguments(
    output_dir=f"{OUT}/checkpoints",
    num_train_epochs=EPOCHS,
    max_steps=10 if SMOKE_TEST else -1,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,                 # float = 5% of all steps
    weight_decay=0.0,
    optim="adamw_torch",
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=5,
    eval_strategy=every, save_strategy=every, eval_steps=10, save_steps=10,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    save_total_limit=2, save_only_model=True,
    remove_unused_columns=False,
    report_to="none",
    seed=SEED, data_seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=ListDataset(train_enc),
                  eval_dataset=ListDataset(dev_enc), data_collator=collate, callbacks=[StopIfNaN()])
t0 = time.time()
trainer.train()
train_minutes = (time.time() - t0) / 60

log = trainer.state.log_history
json.dump(log, open(f"{OUT}/train_log.json", "w"), indent=2)
bad = [x for x in log if "loss" in x and not math.isfinite(x["loss"])]
assert not bad, "Training produced NaN loss - do not use this model. Paste the output to Claude."
print(f"\nTraining took {train_minutes:.1f} min")
print("Dev loss per evaluation:", [(round(x["epoch"], 2), round(x["eval_loss"], 4)) for x in log if "eval_loss" in x])
BEST_CKPT, BEST_DEV_LOSS = trainer.state.best_model_checkpoint, trainer.state.best_metric
print("Best checkpoint (lowest dev loss):", BEST_CKPT, "| dev loss", BEST_DEV_LOSS)

ADAPTER = f"{OUT}/adapter"
trainer.model.save_pretrained(ADAPTER)
tok.save_pretrained(ADAPTER)
print("Saved the LoRA adapter to", ADAPTER)

In [ ]:
# 9) Quick look on 8 DEV examples (not a score - just "did it learn the format?").
model = trainer.model
model.eval()
model.config.use_cache = True
exact = 0
for ex in dev_raw[:8]:
    ids = torch.tensor([prompt_ids(ex["prompt"])], device=model.device)
    with torch.no_grad():
        out = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=256,
                             do_sample=False, eos_token_id=IM_END, pad_token_id=PAD)
    text = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()
    exact += text == ex["answer"]
    print("MODEL :", text[:200])
    print("ANSWER:", ex["answer"][:200], "\n")
print(f"exactly right: {exact}/8  (after only 10 steps in SMOKE_TEST this can be 0 - that's fine)")

In [ ]:
# 10) Merge the adapter into the full model (on CPU, in full precision), then convert to GGUF Q8_0.
del model, trainer
gc.collect(); torch.cuda.empty_cache()
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32, device_map={"": "cpu"})
merged = PeftModel.from_pretrained(base, ADAPTER).merge_and_unload()
merged = merged.to(torch.bfloat16)          # SmolLM3's original storage type
MERGED = f"{OUT}/merged" if not SMOKE_TEST else "/tmp/merged"
merged.save_pretrained(MERGED)
del base, merged
gc.collect()
import shutil
for name in META_FILES:                     # original small files, unchanged -> only weights differ
    src = os.path.join(BASE_FILES, name)
    if os.path.exists(src):
        shutil.copyfile(src, os.path.join(MERGED, name))
print("Merged model saved to", MERGED, "->", sorted(os.listdir(MERGED)))

GGUF = f"{OUT}/smollm3-logtriage-q8_0.gguf" if not SMOKE_TEST else "/tmp/smoke-q8_0.gguf"
run(f"{sys.executable} {LLAMA}/convert_hf_to_gguf.py {MERGED} --outtype q8_0 --outfile {GGUF}")
print(f"GGUF written: {GGUF}  ({os.path.getsize(GGUF) / 1e9:.2f} GB)")

# Safety check: apart from names/file type, every setting in the new GGUF (architecture, RoPE,
# tokenizer, BOS flag, vocabulary) must be identical to the untouched base model's.
FT_META = gguf_meta(GGUF)
diff = sorted(k for k in set(BASE_META) | set(FT_META)
              if not k.startswith("general.") and BASE_META.get(k) != FT_META.get(k))
if diff:
    raise SystemExit(f"GGUF settings differ from the base model: {diff} - paste this to Claude.")
print("GGUF check passed: only the weights differ from the base model.")

In [ ]:
# 11) Record everything needed for the report.
info = {
    "smoke_test": SMOKE_TEST,
    "model_id": MODEL_ID, "gpu": GPU, "data_sha256": DATA_SHA256,
    "add_bos": ADD_BOS, "add_bos_reason": BOS_WHY, "ollama_cross_check": CHECK,
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                 "peft": peft.__version__, "llama_cpp_commit": LLAMA_CPP_COMMIT},
    "settings": {"epochs": EPOCHS, "learning_rate": LEARNING_RATE, "batch": BATCH,
                 "grad_accum": GRAD_ACCUM, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
                 "lora_dropout": LORA_DROPOUT, "max_len": MAX_LEN, "seed": SEED,
                 "quantisation_during_training": "4-bit NF4, double quant, fp16 compute",
                 "lora_targets": "all linear layers", "loss": "answer tokens only"},
    "examples": {"train": len(train_enc), "dev": len(dev_enc),
                 "train_dropped_too_long": train_dropped, "dev_dropped_too_long": dev_dropped},
    "train_minutes": round(train_minutes, 1),
    "dev_loss_by_eval": [(x["epoch"], x["eval_loss"]) for x in log if "eval_loss" in x],
    "best_checkpoint": os.path.basename(str(BEST_CKPT)), "best_dev_loss": BEST_DEV_LOSS,
    "gguf": os.path.basename(GGUF), "gguf_gb": round(os.path.getsize(GGUF) / 1e9, 2),
}
json.dump(info, open(f"{OUT}/run_info.json", "w"), indent=2)
print(json.dumps(info, indent=2))
if SMOKE_TEST:
    print("\nSMOKE TEST PASSED. Now set SMOKE_TEST = False and do Save Version -> Save & Run All (Commit).")
else:
    print("\nDONE. Download from the Output tab: smollm3-logtriage-q8_0.gguf, run_info.json, train_log.json")